###0) Konfiguracja

In [0]:
from datetime import (
    datetime,
    date,
    time,
    timezone,
)
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

dbutils.widgets.text(
    "evaluation_start_date_local",
    "2024-07-01"
)

dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

EVALUATION_START_DATE_TEXT = dbutils.widgets.get(
    "evaluation_start_date_local"
)

EVALUATION_END_DATE_TEXT = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

MARKET_TIMEZONE = "America/Los_Angeles"
LOCAL_TIMEZONE = ZoneInfo(MARKET_TIMEZONE)

evaluation_start_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_START_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

evaluation_end_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_END_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

if evaluation_end_local <= evaluation_start_local:
    raise ValueError(
        "Data końcowa musi być późniejsza "
        "od daty początkowej."
    )

EVALUATION_START_UTC = (
    evaluation_start_local
    .astimezone(timezone.utc)
)

EVALUATION_END_UTC = (
    evaluation_end_local
    .astimezone(timezone.utc)
)

EXPECTED_TARGET_INTERVAL_COUNT = int(
    (
        EVALUATION_END_UTC
        - EVALUATION_START_UTC
    ).total_seconds()
    / (15 * 60)
)

EXPECTED_DECISION_TIME_COUNT = int(
    (
        EVALUATION_END_UTC
        - EVALUATION_START_UTC
    ).total_seconds()
    / 3600
)

print(
    "Zakres Gold:",
    EVALUATION_START_UTC,
    "—",
    EVALUATION_END_UTC
)

print(
    "Oczekiwane interwały docelowe:",
    EXPECTED_TARGET_INTERVAL_COUNT
)

print(
    "Oczekiwane chwile decyzji:",
    EXPECTED_DECISION_TIME_COUNT
)

###1) Kontrola tabel wejściowych

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

INPUT_TABLES = {
    "NLR": (
        "workspace.energy_silver."
        "nlr_decision_features_15m"
    ),
    "WEATHER": (
        "workspace.energy_silver."
        "open_meteo_forecast_asof_hourly"
    ),
    "DAM": (
        "workspace.energy_silver."
        "caiso_dam_lmp"
    ),
    "FMM": (
        "workspace.energy_silver."
        "caiso_fmm_lmp"
    ),
}

for dataset_name, table_name in INPUT_TABLES.items():
    dataframe = spark.table(table_name)

    print("=" * 90)
    print(dataset_name)
    print(f"Tabela: {table_name}")
    print(f"Liczba rekordów: {dataframe.count()}")
    print("Kolumny i typy:")

    for column_name, data_type in dataframe.dtypes:
        print(f"  {column_name}: {data_type}")

###2) Oś decyzji FMM

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

FMM_TABLE = (
    "workspace.energy_silver."
    "caiso_fmm_lmp"
)

fmm_source_df = (
    spark.table(FMM_TABLE)
    .filter(
        (
            F.col("interval_start_utc")
            >= F.lit(
                EVALUATION_START_UTC
            ).cast("timestamp")
        )
        &
        (
            F.col("interval_start_utc")
            < F.lit(
                EVALUATION_END_UTC
            ).cast("timestamp")
        )
    )
)

decision_spine_df = (
    fmm_source_df
    .select(
        F.col("node_id"),
        F.col("rt_bid_gate_close_at_utc").alias(
            "decision_time_utc"
        ),
        F.col("interval_start_utc").alias(
            "target_interval_start_utc"
        ),
        F.col("interval_end_utc").alias(
            "target_interval_end_utc"
        ),
        F.col("price_available_at_utc").alias(
            "fmm_price_available_at_utc"
        ),
        F.col("lmp_price_usd_mwh").alias(
            "actual_fmm_lmp_usd_mwh"
        ),
        F.col("energy_component_usd_mwh").alias(
            "actual_fmm_energy_component_usd_mwh"
        ),
        F.col("congestion_component_usd_mwh").alias(
            "actual_fmm_congestion_component_usd_mwh"
        ),
        F.col("loss_component_usd_mwh").alias(
            "actual_fmm_loss_component_usd_mwh"
        ),
        F.col("is_price_spike").alias(
            "actual_fmm_is_price_spike"
        ),
        F.col("quality_status").alias(
            "fmm_quality_status"
        )
    )
    .withColumn(
        "target_weather_hour_utc",
        F.date_trunc(
            "hour",
            F.col("target_interval_start_utc")
        )
    )
    .withColumn(
        "decision_horizon_minutes",
        (
            F.col("target_interval_start_utc").cast("long")
            - F.col("decision_time_utc").cast("long")
        ) / 60.0
    )
    .withColumn(
        "is_actual_fmm_price_known_at_decision",
        F.col("fmm_price_available_at_utc")
        <= F.col("decision_time_utc")
    )
    .withColumn(
        "decision_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("node_id"),
                F.col(
                    "target_interval_start_utc"
                ).cast("string")
            ),
            256
        )
    )
)

decision_spine_summary_df = (
    decision_spine_df
    .agg(
        F.count("*").alias("target_interval_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.countDistinct("decision_time_utc").alias(
            "distinct_decision_time_count"
        ),
        F.min("decision_time_utc").alias(
            "min_decision_time_utc"
        ),
        F.max("decision_time_utc").alias(
            "max_decision_time_utc"
        ),
        F.min("decision_horizon_minutes").alias(
            "min_horizon_minutes"
        ),
        F.max("decision_horizon_minutes").alias(
            "max_horizon_minutes"
        ),
        F.sum(
            F.col(
                "is_actual_fmm_price_known_at_decision"
            ).cast("int")
        ).alias("price_leakage_count")
    )
)

display(decision_spine_summary_df)

display(
    decision_spine_df
    .select(
        "decision_time_utc",
        "target_interval_start_utc",
        "decision_horizon_minutes",
        "fmm_price_available_at_utc",
        "is_actual_fmm_price_known_at_decision",
        "actual_fmm_lmp_usd_mwh"
    )
    .orderBy("target_interval_start_utc")
    .limit(8)
)

###3) Dane NLR

In [0]:
NLR_TABLE = (
    "workspace.energy_silver."
    "nlr_decision_features_15m"
)

nlr_source_df = (
    spark.table(NLR_TABLE)
    .filter(F.col("system_id") == 2107)
)

# Informacje dostępne w chwili podejmowania decyzji.
nlr_state_df = (
    nlr_source_df
    .select(
        "system_id",
        "decision_time_utc",
        "previous_interval_start_utc",
        "power_available_at_utc",
        F.col("ac_power_previous_15m_kw").alias(
            "observed_ac_power_kw"
        ),
        F.col("pv_energy_previous_15m_mwh").alias(
            "observed_pv_energy_mwh"
        ),
        F.col("previous_power_is_outlier").alias(
            "observed_power_is_outlier"
        ),
        "poa_interval_start_utc",
        "poa_available_at_utc",
        F.col("poa_previous_15m_wm2").alias(
            "observed_poa_wm2"
        ),
        "poa_valid_sample_count",
        "poa_total_sample_count",
        "has_irradiance_outlier",
        "weather_observed_at_utc",
        F.col("ambient_temperature_c").alias(
            "observed_temperature_c"
        ),
        F.col("wind_speed_ms").alias(
            "observed_wind_speed_ms"
        ),
        F.col("wind_direction_deg").alias(
            "observed_wind_direction_deg"
        ),
        "is_temperature_missing",
        "is_wind_speed_missing",
        "is_wind_direction_missing",
        "is_any_weather_outlier",
        F.col("is_core_feature_complete").alias(
            "is_nlr_state_complete"
        )
    )
)

# Rzeczywista produkcja w docelowym interwale.
# To są etykiety do oceny, a nie cechy dostępne dla modelu.
nlr_target_df = (
    nlr_source_df
    .select(
        "target_interval_start_utc",
        F.col("actual_ac_power_next_15m_kw").alias(
            "actual_pv_power_kw"
        ),
        F.col("actual_pv_energy_next_15m_mwh").alias(
            "actual_pv_energy_mwh"
        ),
        F.col("target_power_is_outlier").alias(
            "actual_pv_power_is_outlier"
        ),
        F.col("has_next_interval_target").alias(
            "has_actual_pv_target"
        )
    )
)

pilot_with_nlr_df = (
    decision_spine_df.alias("decision")
    .join(
        nlr_state_df.alias("state"),
        on="decision_time_utc",
        how="left"
    )
    .join(
        nlr_target_df.alias("target"),
        on="target_interval_start_utc",
        how="left"
    )
)

nlr_join_summary_df = (
    pilot_with_nlr_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum(
            F.col("system_id").isNull().cast("int")
        ).alias("missing_nlr_state_count"),
        F.sum(
            F.col("actual_pv_power_kw").isNull().cast("int")
        ).alias("missing_actual_pv_target_count"),
        F.sum(
            (
                F.col("is_nlr_state_complete").isNull()
                | (~F.col("is_nlr_state_complete"))
            ).cast("int")
        ).alias("incomplete_nlr_state_count"),
        F.sum(
            (
                F.col("power_available_at_utc")
                > F.col("decision_time_utc")
            ).cast("int")
        ).alias("power_availability_violation_count"),
        F.sum(
            (
                F.col("poa_available_at_utc")
                > F.col("decision_time_utc")
            ).cast("int")
        ).alias("poa_availability_violation_count"),
        F.sum(
            (
                F.col("target_interval_start_utc")
                <= F.col("decision_time_utc")
            ).cast("int")
        ).alias("target_before_decision_count"),
        F.sum(
            F.col(
                "actual_pv_power_is_outlier"
            ).cast("int")
        ).alias("actual_pv_outlier_count")
    )
)

display(nlr_join_summary_df)

display(
    pilot_with_nlr_df
    .select(
        "decision_time_utc",
        "target_interval_start_utc",
        "decision_horizon_minutes",
        "observed_ac_power_kw",
        "observed_poa_wm2",
        "observed_temperature_c",
        "actual_pv_power_kw",
        "actual_fmm_lmp_usd_mwh"
    )
    .orderBy("target_interval_start_utc")
    .limit(8)
)

###4) Prognoza pogody dostępna w chwili decyzji

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

FULL_WEATHER_TABLE = (
    "workspace.energy_silver."
    "open_meteo_forecast"
)

weather_forecast_candidates_df = (
    spark.table(FULL_WEATHER_TABLE)
    .select(
        F.lit(2107).cast("int").alias("system_id"),
        "forecast_model",
        "forecast_run_id",
        "run_time_utc",
        F.col("available_at_utc").alias(
            "weather_available_at_utc"
        ),
        F.col("forecast_valid_time_utc").alias(
            "weather_valid_time_utc"
        ),
        F.col("temperature_2m_c").alias(
            "forecast_temperature_c"
        ),
        F.col("relative_humidity_2m_pct").alias(
            "forecast_relative_humidity_pct"
        ),
        F.col("cloud_cover_pct").alias(
            "forecast_cloud_cover_pct"
        ),
        F.col("wind_speed_10m_ms").alias(
            "forecast_wind_speed_ms"
        ),
        F.col("wind_direction_10m_deg").alias(
            "forecast_wind_direction_deg"
        ),
        F.col("shortwave_radiation_wm2").alias(
            "forecast_shortwave_radiation_wm2"
        ),
        "forecast_gti_wm2",
        "is_day",
        F.col("is_core_forecast_complete").alias(
            "is_weather_forecast_complete"
        )
    )
    .dropDuplicates([
        "system_id",
        "forecast_run_id",
        "weather_valid_time_utc"
    ])
)

weather_targets_df = (
    decision_spine_df
    .select(
        "decision_id",
        "decision_time_utc",
        "target_interval_start_utc",
        "target_weather_hour_utc"
    )
)

weather_matches_df = (
    weather_targets_df.alias("decision")
    .join(
        weather_forecast_candidates_df.alias("weather"),
        on=(
            (
                F.col("weather.weather_valid_time_utc")
                == F.col(
                    "decision.target_weather_hour_utc"
                )
            )
            &
            (
                F.col("weather.weather_available_at_utc")
                <= F.col("decision.decision_time_utc")
            )
        ),
        how="left"
    )
    .select(
        F.col("decision.decision_id"),
        F.col("decision.decision_time_utc"),
        F.col("decision.target_interval_start_utc"),
        F.col("decision.target_weather_hour_utc"),
        F.col("weather.system_id").alias(
            "weather_system_id"
        ),
        F.col("weather.forecast_model"),
        F.col("weather.forecast_run_id"),
        F.col("weather.run_time_utc").alias(
            "weather_run_time_utc"
        ),
        F.col("weather.weather_available_at_utc"),
        F.col("weather.weather_valid_time_utc"),
        F.col("weather.forecast_temperature_c"),
        F.col("weather.forecast_relative_humidity_pct"),
        F.col("weather.forecast_cloud_cover_pct"),
        F.col("weather.forecast_wind_speed_ms"),
        F.col("weather.forecast_wind_direction_deg"),
        F.col("weather.forecast_shortwave_radiation_wm2"),
        F.col("weather.forecast_gti_wm2"),
        F.col("weather.is_day").alias(
            "forecast_is_day"
        ),
        F.col("weather.is_weather_forecast_complete")
    )
)

weather_selection_window = (
    Window
    .partitionBy("decision_id")
    .orderBy(
        F.col("weather_available_at_utc")
        .desc_nulls_last(),
        F.col("weather_run_time_utc")
        .desc_nulls_last()
    )
)

selected_weather_df = (
    weather_matches_df
    .withColumn(
        "_weather_selection_rank",
        F.row_number().over(weather_selection_window)
    )
    .filter(F.col("_weather_selection_rank") == 1)
    .drop("_weather_selection_rank")
    .withColumn(
        "weather_forecast_age_minutes",
        (
            F.col("decision_time_utc").cast("long")
            - F.col(
                "weather_available_at_utc"
            ).cast("long")
        ) / 60.0
    )
    .withColumn(
        "weather_lead_time_hours",
        (
            F.col("weather_valid_time_utc").cast("long")
            - F.col("weather_run_time_utc").cast("long")
        ) / 3600.0
    )
    .withColumn(
        "weather_target_offset_minutes",
        (
            F.col(
                "target_interval_start_utc"
            ).cast("long")
            - F.col(
                "weather_valid_time_utc"
            ).cast("long")
        ) / 60.0
    )
)

pilot_with_weather_df = (
    pilot_with_nlr_df
    .join(
        selected_weather_df
        .drop(
            "decision_time_utc",
            "target_interval_start_utc",
            "target_weather_hour_utc"
        ),
        on="decision_id",
        how="left"
    )
)

weather_join_summary_df = (
    pilot_with_weather_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum(
            F.col("forecast_run_id").isNull().cast("int")
        ).alias("missing_weather_forecast_count"),
        F.sum(
            (
                F.col("forecast_run_id").isNotNull()
                &
                (
                    F.col("is_weather_forecast_complete")
                    .isNull()
                    |
                    (~F.col(
                        "is_weather_forecast_complete"
                    ))
                )
            ).cast("int")
        ).alias(
            "incomplete_weather_forecast_count"
        ),
        F.sum(
            (
                F.col("weather_available_at_utc")
                > F.col("decision_time_utc")
            ).cast("int")
        ).alias(
            "weather_availability_violation_count"
        ),
        F.sum(
            (
                F.col("weather_valid_time_utc")
                != F.col("target_weather_hour_utc")
            ).cast("int")
        ).alias("weather_valid_time_mismatch_count"),
        F.min("weather_forecast_age_minutes").alias(
            "min_forecast_age_minutes"
        ),
        F.max("weather_forecast_age_minutes").alias(
            "max_forecast_age_minutes"
        ),
        F.min("weather_lead_time_hours").alias(
            "min_weather_lead_hours"
        ),
        F.max("weather_lead_time_hours").alias(
            "max_weather_lead_hours"
        )
    )
)

display(weather_join_summary_df)

display(
    pilot_with_weather_df
    .select(
        "decision_time_utc",
        "target_interval_start_utc",
        "forecast_run_id",
        "weather_available_at_utc",
        "weather_valid_time_utc",
        "forecast_gti_wm2",
        "forecast_temperature_c",
        "actual_pv_power_kw"
    )
    .orderBy("target_interval_start_utc")
    .limit(8)
)

###4A) Pełna tabela prognoz

In [0]:
FULL_WEATHER_TABLE = (
    "workspace.energy_silver."
    "open_meteo_forecast"
)

full_weather_df = spark.table(FULL_WEATHER_TABLE)

print(f"Liczba wszystkich prognoz: {full_weather_df.count()}")
print("Kolumny:")

for column_name, data_type in full_weather_df.dtypes:
    print(f"  {column_name}: {data_type}")

print("Prognozy dla pierwszej godziny pilotażu:")

display(
    full_weather_df
    .filter(
        F.col("forecast_valid_time_utc")
        == F.to_timestamp(
            F.lit(EVALUATION_START_UTC).cast("timestamp")
        )
    )
    .select(
        "forecast_run_id",
        "run_time_utc",
        "available_at_utc",
        "forecast_valid_time_utc",
        "forecast_gti_wm2",
        "temperature_2m_c",
        "is_core_forecast_complete"
    )
    .orderBy("available_at_utc")
)

###5) Dołączenie ceny DAM

In [0]:
DAM_TABLE = (
    "workspace.energy_silver."
    "caiso_dam_lmp"
)

dam_features_df = (
    spark.table(DAM_TABLE)
    .select(
        F.col("interval_start_utc").alias(
            "dam_interval_start_utc"
        ),
        F.col("interval_end_utc").alias(
            "dam_interval_end_utc"
        ),
        F.col("price_available_at_utc").alias(
            "dam_price_available_at_utc"
        ),
        F.col("lmp_price_usd_mwh").alias(
            "dam_lmp_usd_mwh"
        ),
        F.col("energy_component_usd_mwh").alias(
            "dam_energy_component_usd_mwh"
        ),
        F.col("congestion_component_usd_mwh").alias(
            "dam_congestion_component_usd_mwh"
        ),
        F.col("loss_component_usd_mwh").alias(
            "dam_loss_component_usd_mwh"
        ),
        F.col("ghg_component_usd_mwh").alias(
            "dam_ghg_component_usd_mwh"
        ),
        F.col("is_price_spike").alias(
            "dam_is_price_spike"
        ),
        F.col("quality_status").alias(
            "dam_quality_status"
        )
    )
)

pilot_with_markets_df = (
    pilot_with_weather_df.alias("pilot")
    .join(
        dam_features_df.alias("dam"),
        on=(
            F.col("pilot.target_weather_hour_utc")
            == F.col("dam.dam_interval_start_utc")
        ),
        how="left"
    )
    .select(
        F.col("pilot.*"),
        F.col("dam.dam_interval_start_utc"),
        F.col("dam.dam_interval_end_utc"),
        F.col("dam.dam_price_available_at_utc"),
        F.col("dam.dam_lmp_usd_mwh"),
        F.col("dam.dam_energy_component_usd_mwh"),
        F.col("dam.dam_congestion_component_usd_mwh"),
        F.col("dam.dam_loss_component_usd_mwh"),
        F.col("dam.dam_ghg_component_usd_mwh"),
        F.col("dam.dam_is_price_spike"),
        F.col("dam.dam_quality_status")
    )
    .withColumn(
        "is_dam_price_known_at_decision",
        F.col("dam_price_available_at_utc")
        <= F.col("decision_time_utc")
    )
    .withColumn(
        "actual_fmm_minus_dam_spread_usd_mwh",
        F.col("actual_fmm_lmp_usd_mwh")
        - F.col("dam_lmp_usd_mwh")
    )
)

market_join_summary_df = (
    pilot_with_markets_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum(
            F.col("dam_lmp_usd_mwh").isNull().cast("int")
        ).alias("missing_dam_price_count"),
        F.sum(
            F.col(
                "is_dam_price_known_at_decision"
            ).cast("int")
        ).alias("dam_price_known_at_decision_count"),
        F.sum(
            (
                F.col("dam_price_available_at_utc")
                > F.col("decision_time_utc")
            ).cast("int")
        ).alias("dam_availability_violation_count"),
        F.sum(
            (
                F.col("dam_interval_start_utc")
                != F.col("target_weather_hour_utc")
            ).cast("int")
        ).alias("dam_hour_mismatch_count"),
        F.sum(
            (
                F.col("fmm_price_available_at_utc")
                <= F.col("decision_time_utc")
            ).cast("int")
        ).alias("fmm_price_leakage_count"),
        F.sum(
            (
                F.col("dam_quality_status") != "VALID"
            ).cast("int")
        ).alias("invalid_dam_quality_count"),
        F.min("dam_lmp_usd_mwh").alias(
            "minimum_dam_lmp"
        ),
        F.max("dam_lmp_usd_mwh").alias(
            "maximum_dam_lmp"
        ),
        F.min(
            "actual_fmm_minus_dam_spread_usd_mwh"
        ).alias("minimum_fmm_dam_spread"),
        F.max(
            "actual_fmm_minus_dam_spread_usd_mwh"
        ).alias("maximum_fmm_dam_spread")
    )
)

display(market_join_summary_df)

display(
    pilot_with_markets_df
    .select(
        "decision_time_utc",
        "target_interval_start_utc",
        "dam_price_available_at_utc",
        "is_dam_price_known_at_decision",
        "dam_lmp_usd_mwh",
        "actual_fmm_lmp_usd_mwh",
        "actual_fmm_minus_dam_spread_usd_mwh"
    )
    .orderBy("target_interval_start_utc")
    .limit(8)
)

###6) Historia cen FMM dostępna point-in-time

In [0]:
FMM_HISTORY_TABLE = (
    "workspace.energy_silver."
    "caiso_fmm_lmp"
)

fmm_history_source_df = (
    spark.table(FMM_HISTORY_TABLE)
    .filter(F.col("quality_status") == "VALID")
    .select(
        F.col("interval_start_utc").alias(
            "history_fmm_interval_start_utc"
        ),
        F.col("price_available_at_utc").alias(
            "history_fmm_price_available_at_utc"
        ),
        F.col("lmp_price_usd_mwh").alias(
            "history_fmm_lmp_usd_mwh"
        )
    )
)

history_decisions_df = (
    pilot_with_markets_df
    .select(
        "decision_id",
        "decision_time_utc",
        "target_interval_start_utc"
    )
)

fmm_history_matches_df = (
    history_decisions_df.alias("decision")
    .join(
        fmm_history_source_df.alias("history"),
        on=(
            F.col(
                "history.history_fmm_price_available_at_utc"
            )
            <= F.col("decision.decision_time_utc")
        ),
        how="left"
    )
    .select(
        F.col("decision.decision_id"),
        F.col("decision.decision_time_utc"),
        F.col("decision.target_interval_start_utc"),
        F.col("history.history_fmm_interval_start_utc"),
        F.col(
            "history.history_fmm_price_available_at_utc"
        ),
        F.col("history.history_fmm_lmp_usd_mwh")
    )
)

fmm_history_window = (
    Window
    .partitionBy("decision_id")
    .orderBy(
        F.col("history_fmm_price_available_at_utc")
        .desc_nulls_last(),
        F.col("history_fmm_interval_start_utc")
        .desc_nulls_last()
    )
)

ranked_fmm_history_df = (
    fmm_history_matches_df
    .withColumn(
        "_history_rank",
        F.row_number().over(fmm_history_window)
    )
)

fmm_history_features_df = (
    ranked_fmm_history_df
    .filter(F.col("_history_rank") <= 4)
    .groupBy("decision_id")
    .agg(
        F.max(
            F.when(
                F.col("_history_rank") == 1,
                F.col("history_fmm_interval_start_utc")
            )
        ).alias("last_known_fmm_interval_start_utc"),
        F.max(
            F.when(
                F.col("_history_rank") == 1,
                F.col(
                    "history_fmm_price_available_at_utc"
                )
            )
        ).alias(
            "last_known_fmm_price_available_at_utc"
        ),
        F.max(
            F.when(
                F.col("_history_rank") == 1,
                F.col("history_fmm_lmp_usd_mwh")
            )
        ).alias("last_known_fmm_lmp_usd_mwh"),
        F.avg(
            F.when(
                F.col("_history_rank") <= 4,
                F.col("history_fmm_lmp_usd_mwh")
            )
        ).alias("mean_last_4_known_fmm_lmp_usd_mwh"),
        F.min(
            F.when(
                F.col("_history_rank") <= 4,
                F.col("history_fmm_lmp_usd_mwh")
            )
        ).alias("min_last_4_known_fmm_lmp_usd_mwh"),
        F.max(
            F.when(
                F.col("_history_rank") <= 4,
                F.col("history_fmm_lmp_usd_mwh")
            )
        ).alias("max_last_4_known_fmm_lmp_usd_mwh"),
        F.sum(
            F.when(
                (
                    F.col("_history_rank") <= 4
                )
                &
                F.col(
                    "history_fmm_lmp_usd_mwh"
                ).isNotNull(),
                F.lit(1)
            ).otherwise(F.lit(0))
        ).cast("int").alias(
            "known_fmm_price_count_last_4"
        )
    )
)

pilot_with_market_history_df = (
    pilot_with_markets_df
    .join(
        fmm_history_features_df,
        on="decision_id",
        how="left"
    )
    .withColumn(
        "last_known_fmm_price_age_minutes",
        (
            F.col("decision_time_utc").cast("long")
            - F.col(
                "last_known_fmm_price_available_at_utc"
            ).cast("long")
        ) / 60.0
    )
    .withColumn(
        "last_known_fmm_to_target_gap_minutes",
        (
            F.col(
                "target_interval_start_utc"
            ).cast("long")
            - F.col(
                "last_known_fmm_interval_start_utc"
            ).cast("long")
        ) / 60.0
    )
)

fmm_history_summary_df = (
    pilot_with_market_history_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum(
            F.col(
                "last_known_fmm_lmp_usd_mwh"
            ).isNull().cast("int")
        ).alias("missing_last_known_fmm_count"),
        F.sum(
            (
                F.col(
                    "last_known_fmm_price_available_at_utc"
                )
                > F.col("decision_time_utc")
            ).cast("int")
        ).alias("fmm_history_leakage_count"),
        F.sum(
            (
                F.col("last_known_fmm_interval_start_utc")
                >= F.col("target_interval_start_utc")
            ).cast("int")
        ).alias(
            "history_not_before_target_count"
        ),
        F.min(
            "known_fmm_price_count_last_4"
        ).alias("minimum_known_price_count"),
        F.max(
            "known_fmm_price_count_last_4"
        ).alias("maximum_known_price_count"),
        F.min(
            "last_known_fmm_price_age_minutes"
        ).alias("minimum_price_age_minutes"),
        F.max(
            "last_known_fmm_price_age_minutes"
        ).alias("maximum_price_age_minutes")
    )
)

display(fmm_history_summary_df)

display(
    pilot_with_market_history_df
    .select(
        "decision_time_utc",
        "target_interval_start_utc",
        "last_known_fmm_interval_start_utc",
        "last_known_fmm_price_available_at_utc",
        "last_known_fmm_lmp_usd_mwh",
        "mean_last_4_known_fmm_lmp_usd_mwh",
        "known_fmm_price_count_last_4",
        "actual_fmm_lmp_usd_mwh"
    )
    .orderBy("target_interval_start_utc")
    .limit(12)
)

###7) Jakość i gotowość rekordów Gold

In [0]:
MARKET_TIMEZONE = "America/Los_Angeles"

gold_pilot_df = (
    pilot_with_market_history_df
    .withColumn(
        "decision_time_local",
        F.from_utc_timestamp(
            F.col("decision_time_utc"),
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "target_interval_start_local",
        F.from_utc_timestamp(
            F.col("target_interval_start_utc"),
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "target_interval_end_local",
        F.from_utc_timestamp(
            F.col("target_interval_end_utc"),
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "target_delivery_date_local",
        F.to_date("target_interval_start_local")
    )
    .withColumn(
        "target_hour_local",
        F.hour("target_interval_start_local")
    )
    .withColumn(
        "target_quarter",
        (
            F.minute("target_interval_start_local") / 15
        ).cast("int") + F.lit(1)
    )
    .withColumn(
        "target_day_of_week",
        F.dayofweek("target_interval_start_local")
    )
    .withColumn(
        "is_feature_complete",
        (
            F.coalesce(
                F.col("is_nlr_state_complete"),
                F.lit(False)
            )
            &
            F.col("forecast_run_id").isNotNull()
            &
            F.coalesce(
                F.col("is_weather_forecast_complete"),
                F.lit(False)
            )
            &
            F.col("dam_lmp_usd_mwh").isNotNull()
            &
            F.coalesce(
                F.col("is_dam_price_known_at_decision"),
                F.lit(False)
            )
            &
            F.col(
                "last_known_fmm_lmp_usd_mwh"
            ).isNotNull()
        )
    )
    .withColumn(
        "is_label_complete",
        (
            F.col("actual_pv_power_kw").isNotNull()
            &
            F.col("actual_pv_energy_mwh").isNotNull()
            &
            F.col("actual_fmm_lmp_usd_mwh").isNotNull()
            &
            (
                F.col("fmm_quality_status")
                == F.lit("VALID")
            )
            &
            (
                ~F.coalesce(
                    F.col("actual_pv_power_is_outlier"),
                    F.lit(True)
                )
            )
        )
    )
    .withColumn(
        "is_model_ready",
        F.col("is_feature_complete")
        & F.col("is_label_complete")
    )
    .withColumn(
        "feature_quality_status",
        F.when(
            F.col("system_id").isNull(),
            F.lit("MISSING_NLR_STATE")
        )
        .when(
            ~F.coalesce(
                F.col("is_nlr_state_complete"),
                F.lit(False)
            ),
            F.lit("INCOMPLETE_NLR_STATE")
        )
        .when(
            F.col("forecast_run_id").isNull(),
            F.lit("MISSING_WEATHER_FORECAST")
        )
        .when(
            ~F.coalesce(
                F.col("is_weather_forecast_complete"),
                F.lit(False)
            ),
            F.lit("INCOMPLETE_WEATHER_FORECAST")
        )
        .when(
            F.col("dam_lmp_usd_mwh").isNull(),
            F.lit("MISSING_DAM_PRICE")
        )
        .when(
            ~F.coalesce(
                F.col("is_dam_price_known_at_decision"),
                F.lit(False)
            ),
            F.lit("DAM_NOT_AVAILABLE_AT_DECISION")
        )
        .when(
            F.col(
                "last_known_fmm_lmp_usd_mwh"
            ).isNull(),
            F.lit("MISSING_FMM_HISTORY")
        )
        .otherwise(F.lit("READY"))
    )
    .withColumn(
        "label_quality_status",
        F.when(
            F.col("actual_pv_power_kw").isNull(),
            F.lit("MISSING_PV_TARGET")
        )
        .when(
            F.col("actual_pv_energy_mwh").isNull(),
            F.lit("MISSING_PV_ENERGY_TARGET")
        )
        .when(
            F.coalesce(
                F.col("actual_pv_power_is_outlier"),
                F.lit(True)
            ),
            F.lit("PV_TARGET_OUTLIER")
        )
        .when(
            F.col("actual_fmm_lmp_usd_mwh").isNull(),
            F.lit("MISSING_FMM_TARGET")
        )
        .when(
            F.col("fmm_quality_status") != "VALID",
            F.lit("INVALID_FMM_TARGET")
        )
        .otherwise(F.lit("VALID"))
    )
)

gold_readiness_summary_df = (
    gold_pilot_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum(
            F.col("is_feature_complete").cast("int")
        ).alias("feature_complete_count"),
        F.sum(
            F.col("is_label_complete").cast("int")
        ).alias("label_complete_count"),
        F.sum(
            F.col("is_model_ready").cast("int")
        ).alias("model_ready_count"),
        F.sum(
            (~F.col("is_model_ready")).cast("int")
        ).alias("not_model_ready_count"),
        F.min("decision_time_utc").alias(
            "min_decision_time_utc"
        ),
        F.max("target_interval_end_utc").alias(
            "max_target_interval_end_utc"
        )
    )
)

display(gold_readiness_summary_df)

display(
    gold_pilot_df
    .groupBy(
        "feature_quality_status",
        "label_quality_status"
    )
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_time_utc").alias(
            "distinct_decision_time_count"
        )
    )
    .orderBy(
        F.desc("row_count"),
        "feature_quality_status"
    )
)

###8) Testy jakości i data leakage

In [0]:
gold_tests_df = (
    gold_pilot_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        (
            F.count("*")
            - F.countDistinct("decision_id")
        ).alias("duplicate_decision_id_count"),
        F.sum(
            F.col("decision_id").isNull().cast("int")
        ).alias("missing_decision_id_count"),
        F.sum(
            (
                ~F.col("decision_horizon_minutes").isin(
                    75.0, 90.0, 105.0, 120.0
                )
            ).cast("int")
        ).alias("invalid_decision_horizon_count"),
        F.sum(
            (
                F.col("target_interval_start_utc")
                <= F.col("decision_time_utc")
            ).cast("int")
        ).alias("target_not_after_decision_count"),
        F.sum(
            (
                F.col("power_available_at_utc")
                > F.col("decision_time_utc")
            ).cast("int")
        ).alias("power_leakage_count"),
        F.sum(
            (
                F.col("poa_available_at_utc")
                > F.col("decision_time_utc")
            ).cast("int")
        ).alias("irradiance_leakage_count"),
        F.sum(
            (
                F.col("weather_available_at_utc").isNotNull()
                &
                (
                    F.col("weather_available_at_utc")
                    > F.col("decision_time_utc")
                )
            ).cast("int")
        ).alias("weather_leakage_count"),
        F.sum(
            (
                F.col("dam_price_available_at_utc")
                > F.col("decision_time_utc")
            ).cast("int")
        ).alias("dam_price_leakage_count"),
        F.sum(
            (
                F.col(
                    "last_known_fmm_price_available_at_utc"
                ).isNotNull()
                &
                (
                    F.col(
                        "last_known_fmm_price_available_at_utc"
                    )
                    > F.col("decision_time_utc")
                )
            ).cast("int")
        ).alias("fmm_history_leakage_count"),
        F.sum(
            (
                F.col("fmm_price_available_at_utc")
                <= F.col("decision_time_utc")
            ).cast("int")
        ).alias("future_fmm_price_known_count"),
        F.sum(
            (
                ~F.col("is_label_complete")
            ).cast("int")
        ).alias("invalid_label_count"),
        F.sum(
            (
                F.col("is_model_ready")
                != (
                    F.col("is_feature_complete")
                    & F.col("is_label_complete")
                )
            ).cast("int")
        ).alias("model_ready_logic_error_count"),
        F.sum(
            (
                ~F.col("target_quarter").isin(1, 2, 3, 4)
            ).cast("int")
        ).alias("invalid_target_quarter_count"),
        F.sum(
            F.col("is_model_ready").cast("int")
        ).alias("model_ready_count")
    )
)

display(gold_tests_df)

test_result = gold_tests_df.first().asDict()

expected_zero_tests = [
    "duplicate_decision_id_count",
    "missing_decision_id_count",
    "invalid_decision_horizon_count",
    "target_not_after_decision_count",
    "power_leakage_count",
    "irradiance_leakage_count",
    "weather_leakage_count",
    "dam_price_leakage_count",
    "fmm_history_leakage_count",
    "future_fmm_price_known_count",
    "invalid_label_count",
    "model_ready_logic_error_count",
    "invalid_target_quarter_count",
]

failed_tests = [
    test_name
    for test_name in expected_zero_tests
    if test_result[test_name] != 0
]

if (
    test_result["row_count"]
    != EXPECTED_TARGET_INTERVAL_COUNT
):
    failed_tests.append("row_count")

if (
    test_result[
        "distinct_decision_id_count"
    ]
    != EXPECTED_TARGET_INTERVAL_COUNT
):
    failed_tests.append(
        "distinct_decision_id_count"
    )

if test_result["model_ready_count"] <= 0:
    failed_tests.append("model_ready_count")

if failed_tests:
    raise RuntimeError(
        "Nie zaliczono testów Gold: "
        + ", ".join(failed_tests)
    )

print("Wszystkie testy jakości Gold zostały zaliczone.")
print(
    "Rekordy gotowe do modelowania:",
    test_result["model_ready_count"]
)

###9) Zapis tabeli Gold

In [0]:
GOLD_TARGET_TABLE = (
    "workspace.energy_gold."
    "market_decision_replay_15m"
)

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "workspace.energy_gold"
)

gold_output_df = (
    gold_pilot_df
    .withColumn(
        "_pipeline_version",
        F.lit("monthly_v2")
    )
    .withColumn(
        "_evaluation_start_utc",
        F.lit(
            EVALUATION_START_UTC
        ).cast("timestamp")
    )
    .withColumn(
        "_evaluation_end_utc",
        F.lit(
            EVALUATION_END_UTC
        ).cast("timestamp")
    )
    .withColumn(
        "_evaluation_start_date_local",
        F.lit(
            EVALUATION_START_DATE_TEXT
        )
    )
    .withColumn(
        "_evaluation_end_date_local_exclusive",
        F.lit(
            EVALUATION_END_DATE_TEXT
        )
    )
    .withColumn(
        "_gold_created_at_utc",
        F.current_timestamp()
    )
)

(
    gold_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TARGET_TABLE)
)

saved_gold_df = spark.table(GOLD_TARGET_TABLE)

print(f"Zapisano tabelę: {GOLD_TARGET_TABLE}")
print(f"Liczba rekordów: {saved_gold_df.count()}")

display(
    saved_gold_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum(
            F.col("is_model_ready").cast("int")
        ).alias("model_ready_count"),
        F.sum(
            (~F.col("is_model_ready")).cast("int")
        ).alias("not_model_ready_count"),
        F.min("target_interval_start_utc").alias(
            "min_target_interval_utc"
        ),
        F.max("target_interval_end_utc").alias(
            "max_target_interval_utc"
        )
    )
)